<a href="https://colab.research.google.com/github/gd-Sahat/ClockBiasPINN/blob/main/smamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.0 MB/s eta 0:00:00


In [2]:
!apt-get update -qq && apt-get install -qq libomp-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libomp5-14:amd64.
(Reading database ... 126284 files and directories currently installed.)
Preparing to unpack .../libomp5-14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libomp5-14:amd64 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp-14-dev.
Preparing to unpack .../libomp-14-dev_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libomp-14-dev (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp-dev:amd64.
Preparing to unpack .../libomp-dev_1%3a14.0-55~exp2_amd64.deb ...
Unpacking libomp-dev:amd64 (1:14.0-55~exp2) ...
Setting up libomp5-14:amd64 (1:14.0.0-1ubuntu1.1) ...
Setting up libomp-14-dev (1:14.0.0-1ubuntu1.1) ...
Setting up libomp-dev:amd64 (1:14.0-55~exp2) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) 

In [3]:
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.8/113.8 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 36.6 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.5-cp311-cp311-linux_x86_64.whl size=423921852 sha256=c9f7e39822b23dc138395c4649129a143fd83b2feb9030c73e4fd6cba86f83ff
  Stored in directory: /root/.cache/pip/wheels/47/98/d8/581017e8df057622dde7b176c957eac17cbff9f3808bd70466
Successfully built mamba-ssm


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from mamba_ssm import Mamba
import joblib
from numpy.lib.stride_tricks import sliding_window_view
from tqdm import tqdm

ImportError: /usr/local/lib/python3.11/dist-packages/selective_scan_cuda.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN3c107WarningC1ESt7variantIJNS0_11UserWarningENS0_18DeprecationWarningEEERKNS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEEb

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
DATA_PATH = '/content/drive/MyDrive/CLOCKBIASPINN/clkbias_diff_updated.gz'

In [ ]:
df = pd.read_csv(DATA_PATH, compression='gzip')

In [ ]:
# If timestamp is not datetime, convert it
if not np.issubdtype(df['timestamp'].dtype, np.datetime64):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
# Create time_ns feature directly from datetime (nanoseconds since epoch)
df['time_ns'] = df['timestamp'].astype(np.int64)
df['hours_since_start'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600 #raw ns cause scaling issue in data

# Encode satellite ID globally using factorize
df['id_enc'] = pd.factorize(df['id'])[0]


In [ ]:
UPDATE_INTERVAL = 7200  # 2 hours in seconds
df['seconds_since_update'] = df['timestamp'].astype(np.int64) // 1e9 % UPDATE_INTERVAL

In [ ]:
# Use np.where for element-wise condition checking
df['update_flag'] = np.where(df['seconds_since_update'] % UPDATE_INTERVAL == 0, 1, 0)

In [ ]:
df['update_sin'] = np.sin(2 * np.pi * df['seconds_since_update'] / UPDATE_INTERVAL)
df['update_cos'] = np.cos(2 * np.pi * df['seconds_since_update'] / UPDATE_INTERVAL)

In [ ]:
# Prepare raw features
raw_features = df[['hours_since_start','bias_diff_ns', 'drift']].values

# Prepare and standardize targets
targets = df['bias_diff_ns'].values.astype(np.float32).reshape(-1, 1)

In [ ]:
# --------------------------
# Hyperparameters
# --------------------------
WINDOW_SIZE   = 96          # look-back length L
HORIZON       = 240         # forecast horizon T (2 hours at 30s sampling => 240 steps)
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4        # optimizer weight decay for regularization
EPOCHS        = 50
D_MODEL       = 32          # reduced feature dimension D for regularization
D_STATE       = 8           # reduced SSM state dimension N
N_LAYERS      = 1           # reduced number of bidirectional Mamba blocks
DROPOUT_RATE  = 0.2         # dropout rate
PATIENCE      = 10          # early stopping patience
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_SAMPLES   = 60000       # limit data for experimentation

In [ ]:
# Limit to MAX_SAMPLES for quick experiments
limit = min(len(raw_features), MAX_SAMPLES)
raw_features = raw_features[:limit]
targets      = targets[:limit]

In [ ]:
# --------------------------
# 2. Scale and split
# --------------------------
N = len(raw_features)
train_end = int(N * 0.8)
val_end   = int(N * 0.9)
scaler_X = StandardScaler(); scaler_Y = StandardScaler()
scaler_X.fit(raw_features[:train_end]); scaler_Y.fit(targets[:train_end])
features_scaled = scaler_X.transform(raw_features)
targets_scaled  = scaler_Y.transform(targets)
features_train, targets_train = features_scaled[:train_end], targets_scaled[:train_end]
features_val,   targets_val   = features_scaled[train_end:val_end], targets_scaled[train_end:val_end]
features_test,  targets_test  = features_scaled[val_end:],    targets_scaled[val_end:]
joblib.dump(scaler_X, 'scaler_X.pkl'); joblib.dump(scaler_Y, 'scaler_Y.pkl')


['scaler_Y.pkl']

In [ ]:
# 3. Dataset and DataLoader
# --------------------------
class HorizonWindowDataset(Dataset):
    def __init__(self, features: np.ndarray, targets: np.ndarray, window_size: int, horizon: int):
        self.features    = features
        self.targets     = targets
        self.window_size = window_size
        self.horizon     = horizon

    def __len__(self):
        return len(self.features) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        X = self.features[
            idx : idx + self.window_size  # (L, V)
        ]
        y = self.targets[
            idx + self.window_size :
            idx + self.window_size + self.horizon  # (T, 1)
        ]
        return torch.from_numpy(X).float(), torch.from_numpy(y).float()

train_ds = HorizonWindowDataset(features_train, targets_train, WINDOW_SIZE, HORIZON)
val_ds   = HorizonWindowDataset(features_val,   targets_val,   WINDOW_SIZE, HORIZON)
test_ds  = HorizonWindowDataset(features_test,  targets_test,  WINDOW_SIZE, HORIZON)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)


In [ ]:
# --------------------------
# 4. Define S-Mamba Model with Dropout
# --------------------------
class SMamba(nn.Module):
    def __init__(self, window_size, num_vars,
                 d_model=D_MODEL, d_state=D_STATE,
                 n_layers=N_LAYERS, horizon=HORIZON,
                 dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.token = nn.Linear(window_size, d_model)
        # Dropout before and after Mamba
        self.dropout = nn.Dropout(dropout_rate)
        self.mamba_fwd = Mamba(d_model=d_model, d_state=d_state)
        self.mamba_bwd = Mamba(d_model=d_model, d_state=d_state)
        # FFN with dropout
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(d_model, d_model), nn.Dropout(dropout_rate),
            nn.LayerNorm(d_model),
        )
        self.proj = nn.Linear(d_model, horizon)
    def forward(self, U_in):
        U = U_in.permute(0, 2, 1)            # (B, V, L)
        U_tok = self.dropout(self.token(U))  # (B, V, D)
        Y_fwd = self.mamba_fwd(U_tok); Y_bwd = torch.flip(self.mamba_bwd(torch.flip(U_tok, [2])), [2])
        U1 = U_tok + Y_fwd + Y_bwd
        U2 = self.ffn(U1)
        U3 = self.proj(U2)                   # (B, V, T)
        return U3.permute(0, 2, 1)
model = SMamba(WINDOW_SIZE, features_train.shape[1]).to(DEVICE)

In [ ]:
# --------------------------
# 5. Setup Optimizer, Scheduler, Early Stopping
# --------------------------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
best_val = float('inf'); epochs_no_improve = 0


In [ ]:
# --------------------------
# 6. Training Loop with Early Stopping
# --------------------------
for epoch in range(1, EPOCHS+1):
    model.train(); total_train=0.0
    for Xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False):
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(); preds = model(Xb); loss = criterion(preds, yb)
        loss.backward(); optimizer.step(); total_train += loss.item()*Xb.size(0)
    train_loss = total_train/len(train_loader.dataset)

    model.eval(); total_val=0.0
    with torch.no_grad():
        for Xb, yb in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            preds = model(Xb); loss = criterion(preds, yb)
            total_val += loss.item()*Xb.size(0)
    val_loss = total_val/len(val_loader.dataset)
    scheduler.step(val_loss)

    print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

    # Early stopping
    if val_loss < best_val - 1e-4:
        best_val = val_loss; epochs_no_improve = 0; torch.save(model.state_dict(), 'best_model.pth')
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}"); break

# --------------------------
# 7. Testing using best model
# --------------------------
model.load_state_dict(torch.load('best_model.pth')); model.eval()
with torch.no_grad():
    total_test=0.0
    for Xb, yb in tqdm(test_loader, desc="Testing", leave=False):
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        preds= model(Xb); loss=criterion(preds, yb); total_test+=loss.item()*Xb.size(0)
    print(f"Test Loss: {total_test/len(test_loader.dataset):.6f}")


Epoch 1/50 [Train]:   0%|          | 0/749 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([64, 24, 1])) that is different to the input size (torch.Size([64, 24, 3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch 1/50 [Train]:  98%|█████████▊| 737/749 [00:06<00:00, 116.63it/s, train_loss=0.0243]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([9, 24, 1])) that is different to the input size (torch.Size([9, 24, 3])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch 1/50 [Val]:  87%|████████▋ | 80/92 [00:00<00:00, 252.50it/s, val_loss=1.35]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss

Epoch 1/50 - Train Loss: 0.101387, Val Loss: 0.856572


Epoch 2/50 - Train Loss: 0.023167, Val Loss: 0.483161


Epoch 3/50 - Train Loss: 0.020371, Val Loss: 1.079264


Epoch 4/50 - Train Loss: 0.018152, Val Loss: 0.591776


...

Epoch 46/50 - Train Loss: 0.007617, Val Loss: 0.251872


Epoch 47/50 - Train Loss: 0.007520, Val Loss: 0.229299


Epoch 48/50 - Train Loss: 0.007814, Val Loss: 0.226551


Epoch 49/50 - Train Loss: 0.007388, Val Loss: 0.282512


Epoch 50/50 - Train Loss: 0.007805, Val Loss: 0.240970
